# ASD Detection System (Multimodal Approach)
## Project Description
This system aims to detect early indicators of Autism Spectrum Disorder (ASD) by analyzing multiple modalities: **Text, Images, Video, and Audio**.
- **Approach:** We use independent pre-trained models for each data type.
- **Fusion Strategy:** Weighted averaging is used to combine the confidence scores of all models.
- **Explainability:** Includes modality contribution analysis to provide transparency in the final decision.

In [ ]:
# Advanced Multimodal ASD Detection System (Text + Audio + Image + Video)
import torch
from transformers import pipeline
from PIL import Image
import cv2
import os
import numpy as np

# -------------------------
# Load Models (Advanced)
# -------------------------
print("Loading advanced models...")

# Advanced Text model (large, ASD-adapted if possible)
text_model = pipeline("text-classification", model="roberta-large-mnli")

# Advanced Image model (ResNet50 or better, use EfficientNet if available)
from torchvision import models, transforms
image_model = models.efficientnet_b3(weights="EfficientNet_B3_Weights.DEFAULT")
image_model.eval()

transform = transforms.Compose([
    transforms.Resize((300, 300)),  # EfficientNet-B3 expects 300x300
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# -------------------------
# TEXT ANALYSIS - Advanced
# -------------------------
def analyze_text(text):
    # Multiple label support; confidence aggregation
    try:
        results = text_model(text)
        # Use weighted score if multiple classes returned
        weighted_score = sum([r["score"] * (1 if r["label"].lower().startswith("entail") else 0.5) for r in results]) / len(results)
        return min(max(weighted_score, 0), 1)
    except Exception as e:
        print(f"Text analysis error: {e}")
        return 0.5

# -------------------------
# IMAGE ANALYSIS - Advanced
# -------------------------
def analyze_image(image_path):
    try:
        img = Image.open(image_path).convert("RGB")
        img = transform(img).unsqueeze(0)

        with torch.no_grad():
            output = image_model(img)
            prob = torch.softmax(output, dim=1).numpy().flatten()
            # For demo: consider 0.5*max + 0.5*average class probability as "ASD-ness" proxy
            score = float(0.5 * prob.max() + 0.5 * prob.mean())
            return min(max(score, 0), 1)
    except Exception as e:
        print(f"Image analysis error: {e}")
        return 0.5

# -------------------------
# VIDEO ANALYSIS (Advanced)
# -------------------------
def extract_face_features(frame):
    # Placeholder: ideally would use face mesh/emotion/eye contact; here use Histogram of Oriented Gradients (HOG)
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    hog = cv2.HOGDescriptor()
    h = hog.compute(gray)
    if h is not None:
        return h.mean()
    else:
        return 0.0

def analyze_video(video_path):
    cap = cv2.VideoCapture(video_path)
    frame_count = 0
    total_face_feature = 0
    success_frames = 0

    prev_gray = None
    motion_sum = 0.0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        # Advanced: combine face features + motion
        face_feature = extract_face_features(frame)
        total_face_feature += face_feature
        success_frames += 1

        # Optical flow motion detection for richer motion features
        if prev_gray is not None:
            flow = cv2.calcOpticalFlowFarneback(prev_gray, gray, None, 0.5, 3, 15, 3, 5, 1.1, 0)
            mag, _ = cv2.cartToPolar(flow[..., 0], flow[..., 1])
            motion_sum += np.mean(mag)

        prev_gray = gray
        frame_count += 1

        if frame_count > 60:  # Analyze up to 60 frames (about 2s at 30fps)
            break

    cap.release()
    if success_frames == 0:
        return 0.5

    # Normalize and combine features for the risk proxy
    normalized_face = total_face_feature / success_frames
    normalized_motion = motion_sum / (frame_count - 1) if frame_count > 1 else 0
    # Advanced fusion: Use weighted sum (example weights)
    video_score = 0.6 * min(1, normalized_face / 10) + 0.4 * min(1, normalized_motion / 5)
    return min(max(video_score, 0), 1)


# -------------------------
# AUDIO ANALYSIS (Advanced)
# -------------------------
def analyze_audio(audio_path):
    """
    Advanced audio analysis: ASD detection proxy using wav2vec2-large, prosody, and voice event metrics.
    """
    import torchaudio
    import torch
    from transformers import Wav2Vec2Processor, Wav2Vec2ForCTC

    # Use a speech recognition (CTC) model and analyze prosody features + text conf
    processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-large-960h")
    model = Wav2Vec2ForCTC.from_pretrained("facebook/wav2vec2-large-960h")
    model.eval()

    # Load audio
    waveform, sample_rate = torchaudio.load(audio_path)
    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)
    if sample_rate != 16000:
        resampler = torchaudio.transforms.Resample(orig_freq=sample_rate, new_freq=16000)
        waveform = resampler(waveform)
        sample_rate = 16000

    # Extract loudness, pause features (advanced proxy)
    energy = waveform.abs().mean().item()
    silence = (waveform.abs() < 0.01).float().mean().item()

    # Predict text for content feature (proxy for pragmatic errors)
    input_values = processor(waveform.squeeze().numpy(), sampling_rate=sample_rate, return_tensors="pt").input_values
    with torch.no_grad():
        logits = model(input_values).logits
        pred_ids = torch.argmax(logits, dim=-1)
    transcription = processor.batch_decode(pred_ids)[0]

    # Combine features; proxy: high energy, low silence, clear speech = lower ASD risk (inverse)
    content_score = max(0.1, 1 - silence)
    prosody_score = min(1.0, 0.5 * (energy + content_score))
    score = prosody_score
    return min(max(score, 0), 1)

# -------------------------
# NEW: INTERPRETATION & EXPLAINABILITY (Bonus Requirement)
# -------------------------

def get_natural_language_explanation(avg_score):
    """Provides a human-readable interpretation of the ASD risk score[cite: 33]."""
    if avg_score > 0.8:
        return "The system detected strong indicators of ASD traits across multiple modalities. This suggests a high need for professional screening."
    elif avg_score > 0.6:
        return "The system noted moderate indicators of ASD traits. While not definitive, further consultation with a specialist is recommended."
    else:
        return "The system did not detect significant indicators of ASD based on the provided data."

def explain_contribution(scores, weights):
    """Bonus: Provides the breakdown of how each model contributed to the final score[cite: 34, 36]."""
    print("\n--- Explainability (Bonus: Feature Contribution) ---")
    modalities = ["Text", "Image", "Video", "Audio"]
    used_weights = weights[:len(scores)]
    
    for i, modality in enumerate(modalities[:len(scores)]):
        contribution = scores[i] * used_weights[i]
        print(f"- {modality}: Score {scores[i]:.2f} (Weight {used_weights[i]}) -> Contribution: {contribution:.2f}")
    print("-----------------------------------------------------")

# -------------------------
# UPDATED FUSION DECISION
# -------------------------
def final_decision(scores):
    weights = [0.3, 0.25, 0.25, 0.2]  # Text, Image, Video, Audio
    used_weights = weights[:len(scores)]
    avg_score = sum([a * b for a, b in zip(scores, used_weights)]) / sum(used_weights)

    if avg_score > 0.8:
        risk = "HIGH"
    elif avg_score > 0.6:
        risk = "MEDIUM"
    else:
        risk = "LOW"
    
    return risk, avg_score, used_weights

c:\laragon\www\Advance ASD Detection\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading advanced models...


In [ ]:
# Install required libraries if not present
%pip install torch torchvision transformers opencv-python numpy torchaudio

import torch
from transformers import pipeline
from PIL import Image
import cv2
import numpy as np
import torchaudio
from torchvision import models, transforms

print("Libraries imported successfully.")

c:\laragon\www\Advance ASD Detection\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Libraries imported successfully.


In [ ]:
# Load Models
text_model = pipeline("text-classification", model="roberta-large-mnli")

# Image model
image_model = models.efficientnet_b3(weights="DEFAULT")
image_model.eval()

transform = transforms.Compose([
    transforms.Resize((300, 300)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

print("All models loaded.")

c:\laragon\www\Advance ASD Detection\venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Taayder\.cache\huggingface\hub\models--roberta-large-mnli. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


In [ ]:
def analyze_text(text):
    results = text_model(text)
    weighted_score = sum([r["score"] * (1 if r["label"].lower().startswith("entail") else 0.5) for r in results]) / len(results)
    return min(max(weighted_score, 0), 1)

# [এখানে ইমেজ, ভিডিও এবং অডিওর বাকি ফাংশনগুলো বসান]

## Interpretation of Results
The system provides a score between 0 and 1. 
- **High Risk (>0.8):** Significant indicators detected.
- **Medium Risk (0.6 - 0.8):** Moderate indicators; professional screening suggested.
- **Low Risk (<0.6):** No significant indicators detected.

In [ ]:
# User Input (উদাহরণস্বরূপ)
user_text = "The child exhibits repetitive behaviors and difficulty in social eye contact."
# image_path = "path/to/image.jpg"
# ... অন্যান্য ইনপুট

# Score collection
scores = [analyze_text(user_text), 0.75, 0.65, 0.70] # উদাহরণের জন্য মান দেওয়া হয়েছে

# Final Decision
risk, avg_score, used_weights = final_decision(scores)
print(f"Final Risk: {risk} (Score: {avg_score:.2f})")
print(get_natural_language_explanation(avg_score))

# Explainability Bonus
explain_contribution(scores, used_weights)